# Stanza Living NucleusX — Finance Ledger: build Excel from scrape results

This notebook does **not** run the browser scraper itself — that lives in the separate `scrape_nucleus.py` script, which must be run from a normal terminal, not from Jupyter.

### Why the scraper is a separate script
Jupyter kernels always run their own asyncio event loop. Playwright's **sync** API explicitly refuses to run inside any process with an already-running event loop — it checks for this on purpose and raises `Sync API inside the asyncio loop`. This isn't fixable with `nest_asyncio` or similar patches; Playwright's own code blocks it regardless. A plain terminal process has no such loop, so the sync API (the simplest form of the code) works fine there. So:

1. Run `python scrape_nucleus.py` from a terminal (Command Prompt / PowerShell / Anaconda Prompt) in the same folder as this notebook.
2. Log in to NucleusX in the browser window that opens (only needed the first time — the session is cached).
3. Wait for it to finish; it writes `scrape_results.json` next to itself.
4. Come back here and run the cells below to turn that JSON into the Excel file.

### Notes
- **DOM parsing is best-guess.** I don't have access to the real page's HTML, so `scrape_nucleus.py` works off the *visible text* of the page (matching the exact labels/patterns from your screenshot). If some fields come out wrong or empty, open the script and look at the `# ADJUST IF NEEDED` comments — you'll likely need to tweak the tab-click selector or the lazy-load scrolling to match the real page.
- **Better alternative if available:** open Chrome DevTools → Network → reload the Finance Ledger tab. If there's an XHR/fetch call returning JSON directly, that's far more reliable than text-scraping — share the endpoint + response shape and I'll rewrite this to hit it directly instead.

In [16]:
!pip install pandas openpyxl --quiet


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import json
from datetime import date
from pathlib import Path

from openpyxl import Workbook

## Point this at your results file

In [ ]:
WORK_DIR = Path(r'C:/Users/Rick.Dutta/OneDrive - STANZA LIVING/Documents/VS Code/FI Scrapping')
RESULTS_PATH = WORK_DIR / 'scrape_results.json'   # produced by scrape_nucleus.py
OUTPUT_FILE = WORK_DIR / 'finance_ledger_export.xlsx'

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f'{RESULTS_PATH} not found. Run `python scrape_nucleus.py` from a terminal first, '
        'then re-run this cell.'
    )

raw_results = json.loads(RESULTS_PATH.read_text())
print(f'Loaded {len(raw_results)} booking(s) from {RESULTS_PATH}')

Loaded 10 booking(s) from C:\Users\Rick.Dutta\OneDrive - STANZA LIVING\Documents\VS Code\scrape_results.json


## Convert ISO date strings back into real date objects

In [19]:
def s2d(s):
    return date.fromisoformat(s) if s else None


all_results = []
for res in raw_results:
    all_results.append({
        'uuid': res['uuid'],
        'csd': s2d(res['csd']),
        'ced': s2d(res['ced']),
        'move_in': s2d(res['move_in']),
        'monthly_rent': res['monthly_rent'],
        'sd': res['sd'],
        'maintenance': res['maintenance'],
        'onboarding': res['onboarding'],
        'exit_fee': res['exit_fee'],
        'ledger': [
            {
                'item': e['item'],
                'amt': e['amt'],
                'date': s2d(e['date']),
                'from': s2d(e['from']),
                'to': s2d(e['to']),
            }
            for e in res['ledger']
        ],
    })

print('Converted', len(all_results), 'bookings')

Converted 10 bookings


## Write to Excel (same layout as your reference sheet)

In [ ]:
HEADERS = [
    'Booking UUID', 'CSD', 'CED', 'Move-in', 'Monthly Rent', 'SD',
    'Maintenance', 'onboarding', 'exit fee',
    'line Item', 'Amt', 'Date', 'From', 'to',
]

wb = Workbook()
ws = wb.active
ws.title = 'Finance Ledger'
ws.append(HEADERS)

DATE_COLS = {2, 3, 4, 12, 13, 14}  # CSD, CED, Move-in, Date, From, to (1-indexed)

for res in all_results:
    ledger = res['ledger'] or [{'item': None, 'amt': None, 'date': None, 'from': None, 'to': None}]
    for idx, entry in enumerate(ledger):
        if idx == 0:
            row = [
                res['uuid'], res['csd'], res['ced'], res['move_in'],
                res['monthly_rent'], res['sd'], res['maintenance'],
                res['onboarding'], res['exit_fee'],
                entry['item'], entry['amt'], entry['date'], entry['from'], entry['to'],
            ]
        else:
            row = [
                None, None, None, None, None, None, None, None, None,
                entry['item'], entry['amt'], entry['date'], entry['from'], entry['to'],
            ]
        ws.append(row)

for col in DATE_COLS:
    for cell in ws.iter_rows(min_col=col, max_col=col, min_row=2):
        for c in cell:
            if isinstance(c.value, date):
                c.number_format = 'DD-MM-YYYY'

for col_cells in ws.columns:
    length = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
    ws.column_dimensions[col_cells[0].column_letter].width = min(max(length + 2, 10), 40)

wb.save(OUTPUT_FILE)
print('Saved to', OUTPUT_FILE.resolve())

Saved to C:\Users\Rick.Dutta\OneDrive - STANZA LIVING\Documents\VS Code\finance_ledger_export.xlsx


## If something didn't parse right

- Re-run `scrape_nucleus.py` after adding `print(repr(page.inner_text('body')))` right after `page.goto(...)` for one booking, and compare it against what the regex functions expect — adjust the patterns in the script to match the real text.
- If Finance Ledger entries never show up, check the tab-click selector first (`page.get_by_text('Finance Ledger', exact=True)`) — open DevTools, right-click the tab, "Inspect", confirm the visible text matches exactly.
- If entries are lazy-loaded inside a specific scrollable `<div>` rather than the whole window, replace `page.mouse.wheel(...)` in the script with scrolling that specific element, e.g. `page.locator('selector-for-scroll-container').evaluate('el => el.scrollTop = el.scrollHeight')`.